# CodeT5-small baseline on real instructions — KAGGLE

Tests one question: **does CodeT5-small, trained on real instructions (format B), actually learn
`generate` — which scored 0% in every earlier run because the old instructions were reverse-engineered
from code instead of being real requests?** No tree position, no AP loss, no architecture comparison —
this is the clean baseline those experiments were missing.

## What's new in this version
**Task-marker tokens are added to the tokenizer as real special tokens**
(`<GENERATE>`, `<REPAIR>`, `<REFACTOR>`, `<CODE>`, `</CODE>`, `<ERROR>`, `</ERROR>`), not left as plain
text that gets split into several ordinary subword pieces. Standard practice confirmed before writing
this: add the tokens, resize the embedding matrix, then **initialize the new rows to the mean of the
existing embedding table** rather than leaving them at the default random initialization — this gives
each new token a stable, neutral starting point instead of noise the model has to overcome from scratch.

## Before running (on kaggle.com)
1. `dataset_formats_v2.parquet` should already be uploaded as a Kaggle Dataset and attached via
   **"Add Input"** in this notebook's sidebar.
2. Settings -> Accelerator -> **GPU T4 x2** (or GPU P100).
3. Settings -> Internet -> **On** (needed for `pip install` and downloading the base model).

## Model, format, and data
- **Model:** `Salesforce/codet5-small` (60M) — fastest to iterate; if real instructions don't help here,
  nothing downstream matters yet.
- **Format:** `tgt_b_canonical` (format B) only — proven cheapest and most reliable in the earlier
  3-format comparison (99%+ fits the 512-token budget, matched or beat format A on every real metric,
  far ahead of the tree format C).
- **Data:** `dataset_formats_v2.parquet` — the output of `finalize_training_dataset_kaggle.ipynb`, built
  from the REWRITTEN instructions (generate rows that still had the old generic/reverse-engineered
  instruction were already dropped during that step).

## Training
`Seq2SeqTrainer`, 7 max epochs with early stopping (patience 3), lr 2e-5 (full-fine-tune scale, not a
LoRA-scale learning rate), batch 8x2, fp16 + gradient checkpointing. The model here is a plain
`T5ForConditionalGeneration` with no custom wrapper, so the standard Trainer works correctly end to
end — unlike the earlier tree-position notebooks, where a custom `nn.Module` wrapper caused real
friction with Trainer internals.

## Evaluation
Same proven, subprocess-isolated harness from the earlier format-comparison notebook: HumanEval (64
problems) and ClassEval (10 problems), each run in its own process with a hard timeout, only using
problems whose reference solution actually passes the same harness first. Plus exact-match and
validity checks on held-out rows of our own dataset, broken down by task — the `generate` numbers
here are what actually answers the question this notebook exists to answer.

## What this notebook does NOT do
No tree position, no AP loss, no skeleton/fill, no golden-I/O capture, no comparison across the other
6 shortlisted models. All of that is deferred until this baseline confirms the instruction fix works.

## Important
**This notebook has not been executed end to end** (GPU training + real model downloads aren't
something this delivery environment can run). Every cell has been syntax-validated, and the parts that
don't need a GPU (the evaluation harness, the report logic) reuse code already proven correct in earlier
sessions. Run it yourself and read Cell 9's verdict.

In [ ]:
# CELL 0 — INSTALL (pinned -- CodeT5's tokenizer config is incompatible with newer transformers)
!pip install -q "transformers==4.46.3" accelerate datasets pandas pyarrow
print('Installed. RESTART THE SESSION NOW (Run -> Restart Session), then run Cell 1 onward.')

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# CodeT5-small baseline on REAL instructions, format B only (canonical code). Tests one thing:
# does the instruction fix actually let the model learn 'generate', which scored 0% before.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set BEFORE torch import

import ast, copy, gc, hashlib, json, math, random, re, signal, subprocess, sys, tempfile, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PARQUET_FILENAME = "dataset_formats_v2.parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK = Path("/kaggle/working/codet5_baseline")
WORK.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = WORK / "results.json"

BASE_MODEL = "Salesforce/codet5-small"
TARGET_FORMAT_COL = "tgt_b_canonical"   # the one format being trained this run -- see notebook intro
FITS_COL = "fits512_b"

# Task-marker tokens currently live in input_text as plain substrings (e.g. "<GENERATE>"), which a
# tokenizer with no knowledge of them splits into several ordinary subword pieces -- the model has
# to reconstruct "this is a generate example" from fragments each time. Adding them as real special
# tokens gives each one its own single embedding the model can attach task-level meaning to directly.
NEW_SPECIAL_TOKENS = ["<GENERATE>", "<REPAIR>", "<REFACTOR>", "<CODE>", "</CODE>", "<ERROR>", "</ERROR>"]

EPOCHS = 7
LR = 2e-5                # full fine-tune learning rate (NOT a LoRA-scale lr -- see project history on
                          # why this distinction matters for a 60M model)
BATCH_SIZE = 8
GRAD_ACCUM = 2            # effective batch 16
MAX_SRC_LEN = 512
MAX_TGT_LEN = 512

HE_N = 64                          # HumanEval problems (same selection rule as earlier runs)
CLASSEVAL_N = 10                   # ClassEval problems
CLASSEVAL_MAX_PROMPT_TOKENS = 450  # skeletons longer than this would be cut off by the encoder window
EVAL_OWN_N = 300                   # held-out test-split rows generated for own-dataset metrics
GEN_BATCH = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import transformers
print(f"Transformers {transformers.__version__} | torch {torch.__version__} | device {DEVICE}")
if DEVICE != "cuda":
    raise RuntimeError(
        "No GPU found. In the right sidebar: Settings -> Accelerator -> GPU T4 x2 (or GPU P100), "
        "and Internet -> On. The session restarts; then run this cell again."
    )
print(f"GPU: {torch.cuda.get_device_name()} ({torch.cuda.mem_get_info()[1]/1e9:.1f} GB)")
print(f"Base model: {BASE_MODEL} | training format: {TARGET_FORMAT_COL} | epochs: {EPOCHS}")

## Load the dataset, keep only rows that fit format B's token budget

In [ ]:
# CELL 2 — FIND + LOAD dataset_formats_v2.parquet, KEEP ONLY ROWS THAT FIT FORMAT B'S BUDGET
candidates = list(KAGGLE_INPUT_DIR.rglob(PARQUET_FILENAME))
if not candidates:
    print(f"Could not find {PARQUET_FILENAME} under {KAGGLE_INPUT_DIR}. What's attached:")
    if KAGGLE_INPUT_DIR.exists():
        for p in KAGGLE_INPUT_DIR.rglob("*"):
            if p.is_file():
                print(f"  {p}")
    raise FileNotFoundError("Attach the dataset with this file via 'Add Input', then re-run this cell.")
src = candidates[0]
print(f"Using: {src} ({src.stat().st_size/1e6:.1f} MB)")

df = pd.read_parquet(src)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

need = ["id", "unit_id", "split", "task", "input_text", TARGET_FORMAT_COL, FITS_COL, "fits512_input",
        f"ok_{TARGET_FORMAT_COL.split('_')[1]}" if TARGET_FORMAT_COL != "tgt_a_code" else "id"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"expected columns missing: {missing}"

mask = df["fits512_input"].astype(bool) & df[FITS_COL].astype(bool)
if "ok_b" in df.columns and TARGET_FORMAT_COL == "tgt_b_canonical":
    mask &= df["ok_b"].astype(bool)
before = len(df)
df = df[mask].reset_index(drop=True)
print(f"\nKept rows that fit the 512-token budget for input and {TARGET_FORMAT_COL}: {before} -> {len(df)}")

train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print("train tasks:", dict(train_df.task.value_counts()))
assert len(train_df) and len(val_df) and len(test_df), "an empty split -- check the Parquet"

## Tokenizer + model, with task-marker special tokens added and mean-initialized

In [ ]:
# CELL 3 — TOKENIZER + MODEL, WITH TASK-MARKER TOKENS ADDED AND PROPERLY INITIALIZED
# Standard practice (confirmed before writing this cell): add tokens -> resize embeddings -> the
# new rows start as random noise, which a 7-epoch run on a few thousand examples may not have
# enough signal to move far from. Initializing them to the MEAN of the existing embeddings instead
# gives each new token a neutral, stable starting point, which it then specializes during training.
from transformers import AutoTokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
vocab_before = len(tokenizer)
n_added = tokenizer.add_special_tokens({"additional_special_tokens": NEW_SPECIAL_TOKENS})
print(f"Tokenizer vocab: {vocab_before} -> {len(tokenizer)} ({n_added} tokens added)")
for t in NEW_SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(t)
    print(f"  {t!r:12s} -> id {tid}  (single token, not split: {tid != tokenizer.unk_token_id})")

model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)
old_embeddings = model.get_input_embeddings().weight.data.clone()
mean_embedding = old_embeddings.mean(dim=0)

model.resize_token_embeddings(len(tokenizer))

with torch.no_grad():
    input_emb = model.get_input_embeddings()
    for t in NEW_SPECIAL_TOKENS:
        tid = tokenizer.convert_tokens_to_ids(t)
        input_emb.weight.data[tid] = mean_embedding.clone()
    # T5 ties the output projection (lm_head) to the input embeddings by default, so this single
    # assignment covers both directions -- confirmed by checking the config flag below.
print(f"tie_word_embeddings: {model.config.tie_word_embeddings} "
      f"(True means the lm_head update above happened automatically)")

total_p = sum(p.numel() for p in model.parameters())
print(f"Total params after resize: {total_p/1e6:.1f}M")
print("New special tokens initialized to the mean of the existing embedding table (not random).")

## Tokenize all three splits

In [ ]:
# CELL 4 — TOKENIZE ALL THREE SPLITS
def tokenize_split(frame, desc):
    inputs = tokenizer(frame["input_text"].tolist(), truncation=True, max_length=MAX_SRC_LEN)
    targets = tokenizer(frame[TARGET_FORMAT_COL].tolist(), truncation=True, max_length=MAX_TGT_LEN)
    from datasets import Dataset
    ds = Dataset.from_dict({
        "input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"],
    })
    print(f"{desc}: {len(ds)} examples tokenized")
    return ds

train_ds = tokenize_split(train_df, "train")
val_ds = tokenize_split(val_df, "val")
test_ds = tokenize_split(test_df, "test")   # used later by our own eval pass, not by the Trainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)
print("\nData collator ready (dynamic padding, labels padded with -100).")

## Train

In [ ]:
# CELL 5 — TRAIN (Seq2SeqTrainer — the model here is a plain T5ForConditionalGeneration, so the
# standard Trainer works correctly; no custom wrapper is involved this run, unlike the earlier
# tree-position experiments where a custom nn.Module caused real problems with Trainer internals)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

args = Seq2SeqTrainingArguments(
    output_dir=str(WORK / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,   # Trainer handles the use_cache=False/True toggle for train vs
                                    # generate automatically when this flag is set -- no manual
                                    # handling needed here, unlike the custom-wrapper notebooks
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Training: {EPOCHS} max epochs, lr={LR}, batch={BATCH_SIZE}x{GRAD_ACCUM}, early stop patience=3")
train_result = trainer.train()
print(f"\nTraining complete. {train_result}")

best_dir = str(WORK / "best_model")
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)
print(f"Saved best checkpoint to {best_dir}")

history = [{k: v for k, v in log.items() if k in ("epoch", "loss", "eval_loss")}
           for log in trainer.state.log_history if "eval_loss" in log or "loss" in log]
(WORK / "train_history.json").write_text(json.dumps(history, indent=1))
print(f"Saved training history ({len(history)} log entries)")

## Evaluation kit (subprocess-isolated, hard timeouts, proven against reference solutions)

In [ ]:
# CELL 4 — EVALUATION KIT
# Every program under test runs in its OWN Python process with a hard time limit and a memory
# cap, so an infinite loop, sys.exit() or runaway allocation in generated code cannot take down
# the notebook. A test only counts as passed if the runner reaches its final line after the
# tests really ran (earlier notebooks counted a test file that was defined but never run).
import resource  # noqa: F401  (used by the child processes)

PRELUDE = ("import resource\n"
           "try:\n    resource.setrlimit(resource.RLIMIT_AS, (3 << 30, 3 << 30))\nexcept Exception:\n    pass\n")

def run_python(files, entry, args=(), timeout=10):
    """Write {name: text} into a fresh temp dir, run `python entry` there. -> (returncode, stdout, stderr)."""
    with tempfile.TemporaryDirectory() as d:
        for name, text in files.items():
            Path(d, name).write_text((PRELUDE + text) if name == entry else text, encoding="utf-8")
        proc = subprocess.Popen([sys.executable, entry, *map(str, args)], cwd=d, stdin=subprocess.DEVNULL,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, encoding="utf-8",
                                errors="replace", start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return proc.returncode, out, err
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                pass
            try:
                proc.communicate(timeout=5)
            except Exception:
                pass
            return -9, "", "timeout"

HE_RUNNER = '''
import json, sys
spec = json.load(open("spec.json"))
src = open("prog.py", encoding="utf-8").read()
ns = {"__name__": "heval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
    ns["check"](ns[spec["entry"]])
except BaseException as e:
    sys.stderr.write(type(e).__name__ + ": " + str(e)[:300] + "\\n")
    sys.exit(1)
print("__PASS__")
'''

GOLDEN_RUNNER = '''
import json, sys
fname = sys.argv[1]
cases = json.load(open("cases.json"))
ns = {"__name__": "gen_module"}
try:
    exec(compile(open("gen.py", encoding="utf-8").read(), "gen.py", "exec"), ns)
except BaseException as e:
    sys.stderr.write("EXEC " + type(e).__name__ + "\\n"); sys.exit(1)
fn = ns.get(fname)
if not callable(fn):
    sys.stderr.write("NO_FUNCTION\\n"); sys.exit(2)
def norm(x):
    try:
        return json.loads(json.dumps(x))
    except Exception:
        return repr(x)
bad = 0
for c in cases:
    try:
        out = fn(**c["input"])
        ok = ("output" in c) and norm(out) == c["output"]
    except BaseException as e:
        ok = ("exception" in c) and type(e).__name__ == c["exception"]
    bad += (not ok)
if bad:
    sys.stderr.write("FAILED %d/%d\\n" % (bad, len(cases))); sys.exit(3)
print("__PASS__")
'''

CE_RUNNER = '''
import io, json, sys, unittest
src = open("gen.py", encoding="utf-8").read() + "\\n\\n" + open("test.py", encoding="utf-8").read()
ns = {"__name__": "classeval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
except BaseException as e:
    print(json.dumps({"status": "exec_fail", "error": (type(e).__name__ + ": " + str(e))[:200]})); sys.exit(0)
classes = [v for v in ns.values() if isinstance(v, type) and issubclass(v, unittest.TestCase) and v is not unittest.TestCase]
total = failed = 0
for cls in classes:
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    total += res.testsRun
    failed += len(res.failures) + len(res.errors)
print(json.dumps({"status": "ok", "tests": total, "failed": failed}))
'''

def run_golden(src, func_name, cases_json, timeout=10):
    rc, out, err = run_python({"runner.py": GOLDEN_RUNNER, "gen.py": src, "cases.json": cases_json},
                              "runner.py", args=[func_name], timeout=timeout)
    return rc == 0 and "__PASS__" in out

def top_level_imports(src):
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return "\n".join(l for l in src.splitlines() if l.startswith(("import ", "from ")))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""

def humaneval_header(prompt, entry):
    """Imports and fully written helper functions from the prompt (everything except the entry point)."""
    try:
        tree = ast.parse(prompt)
        keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))
                or (isinstance(n, ast.FunctionDef) and n.name != entry)]
        return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""
    except SyntaxError:
        return top_level_imports(prompt)

def run_humaneval(prompt, entry, test, gen_src, timeout=20):   # same limit as when the reference solutions are validated
    prog = humaneval_header(prompt, entry) + "\n\n" + gen_src + "\n\n" + test + "\n"
    rc, out, err = run_python({"runner.py": HE_RUNNER, "prog.py": prog, "spec.json": json.dumps({"entry": entry})},
                              "runner.py", timeout=timeout)
    return (rc == 0 and "__PASS__" in out), (err.strip().splitlines()[-1][:160] if err.strip() else ("timeout" if rc == -9 else ""))

def run_classeval(skeleton, test, gen_src, timeout=30):
    gen = top_level_imports(skeleton) + "\n\n" + gen_src
    rc, out, err = run_python({"runner.py": CE_RUNNER, "gen.py": gen, "test.py": test}, "runner.py", timeout=timeout)
    if rc == -9:
        return {"status": "timeout", "tests": 0, "failed": 0}
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {"status": "runner_error", "tests": 0, "failed": 0, "error": err.strip()[-160:]}

def defines(src, name, kind):
    try:
        return any(isinstance(n, kind) and n.name == name for n in ast.walk(ast.parse(src)))
    except Exception:
        return False

# ---------------- benchmark data (Hugging Face first, GitHub as a fallback) ----------------
def load_humaneval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("evalplus/humanevalplus", split="test")]
    import gzip, urllib.request
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz", timeout=60).read()
    return [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]

def load_classeval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("FudanSELab/ClassEval", split="test")]
    import urllib.request
    return json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/FudanSELab/ClassEval/master/data/ClassEval_data.json", timeout=60).read().decode("utf-8"))

HE_REQ = ("task_id", "prompt", "entry_point", "test", "canonical_solution")
CE_REQ = ("task_id", "skeleton", "test", "solution_code", "class_name")
def usable(rows, req):
    """Keep rows that have every field we need as a non-empty string (guards against a differently shaped copy of a benchmark)."""
    return [r for r in rows if all(isinstance(r.get(k), str) and r.get(k) for k in req)]

def canonical_humaneval_src(row):
    tree = ast.parse(row["prompt"] + row["canonical_solution"])
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == row["entry_point"])
    return ast.unparse(node)

def select_humaneval(n, seed):
    """Same selection rule as the earlier runs; keeps only problems whose reference solution passes
    in THIS environment, so a failure can be blamed on the model and not on the setup."""
    best = None
    for source in ("hf", "github"):
        try:
            rows = usable(load_humaneval_rows(source), HE_REQ)
            if not rows:
                raise ValueError("no rows with the expected fields")
        except Exception as e:
            print(f"  HumanEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
            continue
        idx = list(range(len(rows)))
        random.Random(seed).shuffle(idx)
        step = len(idx) / n
        chosen = [rows[i] for i in sorted(idx[int(k * step)] for k in range(n))]
        good = []
        for r in chosen:
            try:
                ok, _ = run_humaneval(r["prompt"], r["entry_point"], r["test"], canonical_humaneval_src(r), timeout=20)
            except Exception:
                ok = False
            if ok:
                good.append(r)
        print(f"  HumanEval from {source}: {len(good)}/{len(chosen)} reference solutions pass the harness")
        if best is None or len(good) > len(best[1]):
            best = (source, good, len(chosen))
        if len(good) >= 0.9 * len(chosen):
            break
    return best

def select_classeval(n, seed, tok):
    for source in ("hf", "github"):
        try:
            rows = usable(load_classeval_rows(source), CE_REQ)
            if not rows:
                raise ValueError("no rows with the expected fields")
            break
        except Exception as e:
            print(f"  ClassEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
    else:
        return None
    fits = [r for r in rows if len(tok(f"<GENERATE> Complete this Python class:\n{r['skeleton']}")["input_ids"]) <= CLASSEVAL_MAX_PROMPT_TOKENS]
    random.Random(seed).shuffle(fits)
    chosen, tried = [], 0
    for r in fits:
        if len(chosen) >= n or tried >= 60:
            break
        tried += 1
        res = run_classeval(r["skeleton"], r["test"], r["solution_code"], timeout=30)
        if res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0:
            chosen.append(r)
    print(f"  ClassEval from {source}: {len(rows)} problems, {len(fits)} fit the encoder window, "
          f"tried {tried}, kept {len(chosen)} whose reference solution passes the harness")
    return source, chosen

print("Evaluation kit ready (isolated processes, hard time limits, tests must actually run).")

## Generation + scoring utilities

In [ ]:
# CELL 7 — GENERATION + SCORING UTILITIES (single format this run, so simpler than the 4-format
# comparison notebook -- no format-conversion step, the model's raw output IS Python already)
@torch.no_grad()
def generate_texts(mdl, texts, max_new_tokens=MAX_TGT_LEN, batch_size=GEN_BATCH):
    """Greedy, single pass (true pass@1 -- no beams, no sampling)."""
    mdl.eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outs = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
        with torch.amp.autocast("cuda", enabled=True):
            seq = mdl.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                               max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
        for j, i in enumerate(idx):
            outs[i] = tokenizer.decode(seq[j], skip_special_tokens=True)
    return outs

def is_valid_python(code_str):
    try:
        tree = ast.parse(code_str)
    except Exception:
        return False, False
    try:
        compile(tree, "<e>", "exec")
        return True, True
    except Exception:
        return True, False

def norm_dump(code_str):
    tree = ast.parse(code_str)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def exact_match(pred_src, ref_src):
    try:
        return norm_dump(pred_src) == norm_dump(ref_src)
    except Exception:
        return False

print("Generation + scoring utilities ready.")

## Run the full evaluation

In [ ]:
# CELL 8 — RUN THE FULL EVALUATION
summary = {}

print("Selecting benchmark problems (reference solutions are run through the harness first)...")
he_sel = None
try:
    he_sel = select_humaneval(HE_N, SEED)
except Exception as e:
    print(f"  HumanEval selection failed ({type(e).__name__}: {str(e)[:100]}); it will be skipped")
he_problems = he_sel[1] if he_sel else []

ce_sel = None
try:
    ce_sel = select_classeval(CLASSEVAL_N, SEED, tokenizer)
except Exception as e:
    print(f"  ClassEval selection failed ({type(e).__name__}: {str(e)[:100]}); it will be skipped")
ce_problems = ce_sel[1] if ce_sel else []

print(f"\nUsable: HumanEval {len(he_problems)}  ClassEval {len(ce_problems)}")

# ---------- own-dataset ----------
random.seed(SEED)
eval_rows = test_df.sample(min(EVAL_OWN_N, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"\nGenerating {len(eval_rows)} own-dataset outputs...")
t0 = time.time()
gens = generate_texts(model, eval_rows["input_text"].tolist())
gen_sec = time.time() - t0
print(f"  done in {gen_sec:.0f}s")

own_records = []
for (_, row), text in zip(eval_rows.iterrows(), gens):
    parses, compiles = is_valid_python(text)
    exact = exact_match(text, row[TARGET_FORMAT_COL]) if compiles else False
    own_records.append({"id": row["id"], "task": row["task"], "valid": compiles, "parses": parses,
                        "exact": exact, "n_tokens": len(tokenizer(text)["input_ids"])})
own_df = pd.DataFrame(own_records)

print("\nSamples (one per task, raw model output):")
for task in ("generate", "repair", "refactor"):
    sub = own_df[own_df.task == task]
    if len(sub):
        i = sub.index[0]
        print(f"  [{task}] valid={own_df.loc[i,'valid']} exact={own_df.loc[i,'exact']}")
        print(f"    {gens[i][:200]!r}")

summary["own"] = {
    "n": len(own_df), "valid": int(own_df.valid.sum()), "exact": int(own_df.exact.sum()),
    "gen_seconds": gen_sec,
    "by_task": {t: {"n": len(g), "valid": int(g.valid.sum()), "exact": int(g.exact.sum())}
                for t, g in own_df.groupby("task")},
}
print(f"\nOwn-dataset: valid {summary['own']['valid']}/{summary['own']['n']}  "
      f"exact {summary['own']['exact']}/{summary['own']['n']}")
for t, s in summary["own"]["by_task"].items():
    print(f"  {t:9s} valid {s['valid']}/{s['n']}  exact {s['exact']}/{s['n']}")

# ---------- HumanEval ----------
he_passed = 0
if he_problems:
    gens_he = generate_texts(model, [f"<GENERATE> Complete this Python function:\n{r['prompt']}" for r in he_problems])
    for r, text in zip(he_problems, gens_he):
        valid, _ = is_valid_python(text)
        has_def = valid and defines(text, r["entry_point"], ast.FunctionDef)
        passed = run_humaneval(r["prompt"], r["entry_point"], r["test"], text)[0] if has_def else False
        he_passed += bool(passed)
    print(f"\nHumanEval pass@1: {he_passed}/{len(he_problems)} ({he_passed/len(he_problems)*100:.1f}%)")
summary["humaneval"] = {"n": len(he_problems), "passed": he_passed}

# ---------- ClassEval ----------
ce_passed = 0
if ce_problems:
    gens_ce = generate_texts(model, [f"<GENERATE> Complete this Python class:\n{r['skeleton']}" for r in ce_problems])
    for r, text in zip(ce_problems, gens_ce):
        valid, _ = is_valid_python(text)
        has_cls = valid and defines(text, r["class_name"], ast.ClassDef)
        res = run_classeval(r["skeleton"], r["test"], text) if has_cls else {"status": "no_class", "tests": 0, "failed": 0}
        passed = res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0
        ce_passed += bool(passed)
    print(f"ClassEval class-level pass: {ce_passed}/{len(ce_problems)}")
summary["classeval"] = {"n": len(ce_problems), "passed": ce_passed}

RESULTS_FILE.write_text(json.dumps(summary, indent=2, default=str))
print(f"\nSaved: {RESULTS_FILE}")

## Report — did the instruction fix work?

In [ ]:
# CELL 9 — REPORT: DID THE INSTRUCTION FIX WORK?
history = json.loads((WORK / "train_history.json").read_text())
summary = json.loads(RESULTS_FILE.read_text())

print("=" * 90)
print("CodeT5-small BASELINE on real instructions (format B, with task-marker special tokens)")
print("=" * 90)

val_losses = [h["eval_loss"] for h in history if "eval_loss" in h]
print(f"\nValidation loss by epoch: {' -> '.join(f'{v:.3f}' for v in val_losses)}")

own = summary["own"]
print(f"\nOwn-dataset (held-out test rows, n={own['n']}):")
print(f"  valid Python: {own['valid']}/{own['n']} ({own['valid']/own['n']*100:.1f}%)")
print(f"  exact match:  {own['exact']}/{own['n']} ({own['exact']/own['n']*100:.1f}%)")
for t, s in own["by_task"].items():
    print(f"    {t:9s} valid {s['valid']}/{s['n']} ({s['valid']/s['n']*100:.1f}%)   "
          f"exact {s['exact']}/{s['n']} ({s['exact']/s['n']*100:.1f}%)")

he = summary["humaneval"]
ce = summary["classeval"]
print(f"\nHumanEval: {he['passed']}/{he['n']}" + (f" ({he['passed']/he['n']*100:.1f}%)" if he["n"] else " (skipped)"))
print(f"ClassEval: {ce['passed']}/{ce['n']}" + (f" ({ce['passed']/ce['n']*100:.1f}%)" if ce["n"] else " (skipped)"))

print("\n" + "=" * 90)
print("VERDICT")
print("=" * 90)
gen_stats = own["by_task"].get("generate")
if gen_stats is None:
    print("No 'generate' rows were in the evaluation sample -- cannot judge the instruction fix from this run.")
else:
    gen_exact_pct = gen_stats["exact"] / gen_stats["n"] * 100
    gen_valid_pct = gen_stats["valid"] / gen_stats["n"] * 100
    he_pct = (he["passed"] / he["n"] * 100) if he["n"] else 0
    if gen_exact_pct > 1 or he_pct > 1:
        print(f"Generate moved off zero: {gen_exact_pct:.1f}% exact match, {he_pct:.1f}% HumanEval.")
        print("The instruction fix appears to have worked. Next: scale this recipe to the other")
        print("shortlisted base models (CodeT5+ 220M/770M, CodeGen-350M, Qwen 0.5B/-Instruct, Maincoder-1B)")
        print("to find which performs best on this corrected dataset.")
    elif gen_valid_pct > 50:
        print(f"Generate still has 0% (or near-0%) exact match and HumanEval, but {gen_valid_pct:.1f}% of its")
        print("outputs are at least valid Python -- the model learned the TASK FORMAT but not correct")
        print("behavior yet. This could mean: needs more epochs/data, or CodeT5-small (60M) is too small")
        print("for this task even with good instructions. Try a stronger base model next (e.g.")
        print("Qwen2.5-Coder-0.5B-Instruct) on the same dataset before concluding architecture is the issue.")
    else:
        print(f"Generate is still near 0% on every metric, including basic validity ({gen_valid_pct:.1f}%).")
        print("This suggests the bottleneck is NOT the instructions anymore (those are confirmed real asks)")
        print("-- it's likely CodeT5-small's capacity. Test the same dataset on a stronger base model next.")
print(f"\nFor reference, repair/refactor already worked reasonably well before this fix "
      f"(74-84% exact match in earlier runs) -- compare this run's repair/refactor numbers above")
print("against that to confirm nothing regressed.")

## Finish (download from Kaggle's Output tab)

In [ ]:
# CELL 10 — FINISH (Kaggle: download from the Output tab)
for f in (RESULTS_FILE, WORK / "train_history.json"):
    print(f"{f}  ({f.stat().st_size/1e3:.0f} KB)")
print(f"Model checkpoint: {WORK / 'best_model'}  (saved, not auto-zipped -- see note below)")
print("\nTo get these files onto your computer:")
print("  1. Click 'Save Version' (top right) -> 'Save & Run All' -> Save.")
print("  2. Once it finishes, open this notebook's 'Output' tab.")
print("  3. results.json and train_history.json will be listed there with a Download button.")
print("  Note: the saved model checkpoint directory is large; download only results.json and")
print("  train_history.json unless you specifically need the model weights themselves.")